## Example 1: scikit_learn functionality test 1

In [ ]:
# Example 1: Basic machine learning pipeline with classification
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Generate synthetic classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    n_classes=3,
    random_state=42
)

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Create and train a logistic regression model
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Model coefficients shape: {clf.coef_.shape}")
print(f"Model intercept: {clf.intercept_}")

# Show prediction probabilities for first 5 samples
print(f"First 5 prediction probabilities:")
for i in range(5):
    print(f"Sample {i}: pred={y_pred[i]}, actual={y_test[i]}, proba={y_pred_proba[i]}")

print("Example 1 completed successfully")

## Example 2: scikit_learn functionality test 2

In [ ]:
# Example 2: Regression with preprocessing and model evaluation
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Generate synthetic regression dataset
X, y = make_regression(
    n_samples=500,
    n_features=15,
    n_informative=10,
    noise=0.1,
    random_state=42
)

print(f"Regression dataset: X={X.shape}, y shape={y.shape}")
print(f"Target statistics: mean={y.mean():.2f}, std={y.std():.2f}")

# Create preprocessing and model pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Test Performance:")
print(f"  MSE: {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  R² Score: {r2:.4f}")

# Cross-validation scores
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

print(f"Cross-validation RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std() * 2:.4f})")

# Feature importance from the random forest
feature_importance = pipeline.named_steps['regressor'].feature_importances_
top_features = np.argsort(feature_importance)[-5:]
print(f"Top 5 important features: {top_features} with importance {feature_importance[top_features]}")

# Test prediction on single sample
single_sample = X_test[:1]
single_pred = pipeline.predict(single_sample)
print(f"Single prediction: input_mean={single_sample.mean():.2f}, prediction={single_pred[0]:.2f}")

print("Example 2 completed successfully")

## Example 3: scikit_learn functionality test 3

In [ ]:
# Example 3: Clustering and dimensionality reduction
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
import numpy as np

# Generate clustering dataset
X, y_true = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=1.5,
    random_state=42
)

print(f"Clustering dataset: X={X.shape}")
print(f"True clusters: {len(np.unique(y_true))}")

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

# K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_scaled)

# DBSCAN clustering
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_scaled)

# Evaluate clustering performance
kmeans_ari = adjusted_rand_score(y_true, y_kmeans)
kmeans_silhouette = silhouette_score(X_scaled, y_kmeans)

# Count DBSCAN clusters (excluding noise points labeled as -1)
dbscan_clusters = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
dbscan_noise = list(y_dbscan).count(-1)

if dbscan_clusters > 1:
    dbscan_ari = adjusted_rand_score(y_true, y_dbscan)
    dbscan_silhouette = silhouette_score(X_scaled, y_dbscan)
else:
    dbscan_ari, dbscan_silhouette = 0, 0

print(f"\nK-Means Results:")
print(f"  Clusters found: {len(np.unique(y_kmeans))}")
print(f"  Adjusted Rand Index: {kmeans_ari:.4f}")
print(f"  Silhouette Score: {kmeans_silhouette:.4f}")
print(f"  Cluster centers shape: {kmeans.cluster_centers_.shape}")

print(f"\nDBSCAN Results:")
print(f"  Clusters found: {dbscan_clusters}")
print(f"  Noise points: {dbscan_noise}")
print(f"  Adjusted Rand Index: {dbscan_ari:.4f}")
print(f"  Silhouette Score: {dbscan_silhouette:.4f}")

# Analyze PCA components
print(f"\nPCA Analysis:")
print(f"First component weights: {pca.components_[0][:5]}")  # First 5 weights
print(f"Second component weights: {pca.components_[1][:5]}")  # First 5 weights

print("Example 3 completed successfully")

## Example 4: scikit_learn functionality test 4

In [ ]:
# Example 4: Advanced preprocessing and feature selection
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import numpy as np

# Generate complex classification dataset
X, y = make_classification(
    n_samples=800,
    n_features=20,
    n_informative=12,
    n_redundant=8,
    n_classes=3,
    class_sep=1.2,
    random_state=42
)

print(f"Original dataset: X={X.shape}, y={y.shape}")

# Add polynomial features (degree 2, interaction features only)
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X)
print(f"After polynomial features: {X_poly.shape}")

# Feature selection using SelectKBest with chi2
# First, make features non-negative for chi2
X_positive = X_poly - X_poly.min() + 1e-6
selector_univariate = SelectKBest(score_func=chi2, k=30)
X_selected_univariate = selector_univariate.fit_transform(X_positive, y)

print(f"After univariate selection: {X_selected_univariate.shape}")
print(f"Selected feature indices: {selector_univariate.get_support(indices=True)[:10]}...")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_selected_univariate, y, test_size=0.2, random_state=42, stratify=y
)

# Recursive Feature Elimination with Gradient Boosting
gb_classifier = GradientBoostingClassifier(random_state=42)
rfe = RFE(estimator=gb_classifier, n_features_to_select=15)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)

print(f"After RFE: training={X_train_rfe.shape}, test={X_test_rfe.shape}")
print(f"RFE selected features: {np.sum(rfe.support_)} out of {len(rfe.support_)}")

# Grid search for hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.2],
    'max_depth': [3, 4]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=1  # Use single thread for reproducibility
)

# Fit grid search on RFE-selected features
grid_search.fit(X_train_rfe, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Make predictions with best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_rfe)

# Detailed evaluation
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Feature importance from the best model
feature_importances = best_model.feature_importances_
top_5_indices = np.argsort(feature_importances)[-5:]
print(f"Top 5 feature importances: {feature_importances[top_5_indices]}")

# Prediction probabilities for first few samples
y_pred_proba = best_model.predict_proba(X_test_rfe[:3])
print(f"\nFirst 3 prediction probabilities:")
for i, proba in enumerate(y_pred_proba):
    print(f"Sample {i}: {proba} -> predicted class {y_pred[i]}")

print("Example 4 completed successfully")

## Example 5: scikit_learn functionality test 5

In [ ]:
# Example 5: Model persistence and production pipeline
from sklearn.datasets import make_classification
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import validation_curve
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pickle
from io import BytesIO

# Custom transformer for demonstration
class CustomFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, add_interaction=True):
        self.add_interaction = add_interaction
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        if self.add_interaction and X.shape[1] >= 2:
            # Add simple interaction feature
            interaction = X[:, 0] * X[:, 1]
            X_transformed = np.column_stack([X_transformed, interaction])
        return X_transformed

# Generate dataset for ensemble learning
X, y = make_classification(
    n_samples=600,
    n_features=12,
    n_informative=8,
    n_classes=2,
    class_sep=1.5,
    random_state=42
)

print(f"Dataset for ensemble: X={X.shape}, y={y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Create individual classifiers
lr = LogisticRegression(random_state=42, max_iter=1000)
rf = RandomForestClassifier(n_estimators=50, random_state=42)
svc = SVC(probability=True, random_state=42, gamma='scale')

# Create voting ensemble with preprocessing pipeline
ensemble_pipeline = Pipeline([
    ('custom_transform', CustomFeatureTransformer(add_interaction=True)),
    ('scaler', StandardScaler()),
    ('voting_clf', VotingClassifier(
        estimators=[('lr', lr), ('rf', rf), ('svc', svc)],
        voting='soft'  # Use probability voting
    ))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the ensemble pipeline
ensemble_pipeline.fit(X_train, y_train)

# Evaluate ensemble performance
train_score = ensemble_pipeline.score(X_train, y_train)
test_score = ensemble_pipeline.score(X_test, y_test)

print(f"Ensemble Performance:")
print(f"  Training accuracy: {train_score:.4f}")
print(f"  Test accuracy: {test_score:.4f}")

# Individual model performance within ensemble
voting_clf = ensemble_pipeline.named_steps['voting_clf']
X_train_transformed = ensemble_pipeline[:-1].transform(X_train)
X_test_transformed = ensemble_pipeline[:-1].transform(X_test)

print(f"Individual model scores on transformed data:")
for name, clf in voting_clf.estimators:
    clf.fit(X_train_transformed, y_train)
    score = clf.score(X_test_transformed, y_test)
    print(f"  {name}: {score:.4f}")

# Validation curve for ensemble complexity analysis
param_range = [10, 30, 50, 70, 100]
train_scores, val_scores = validation_curve(
    RandomForestClassifier(random_state=42),
    X_train_transformed, y_train,
    param_name='n_estimators',
    param_range=param_range,
    cv=3, scoring='accuracy'
)

print(f"Validation curve (RF n_estimators):")
for i, n_est in enumerate(param_range):
    print(f"  {n_est}: train={train_scores[i].mean():.4f}, val={val_scores[i].mean():.4f}")

# Model serialization and deserialization
print(f"\nModel Persistence:")

# Serialize the model
model_buffer = BytesIO()
pickle.dump(ensemble_pipeline, model_buffer)
model_size = len(model_buffer.getvalue())
print(f"Serialized model size: {model_size} bytes")

# Deserialize the model
model_buffer.seek(0)
loaded_model = pickle.load(model_buffer)

# Test loaded model
loaded_predictions = loaded_model.predict(X_test[:5])
original_predictions = ensemble_pipeline.predict(X_test[:5])

print(f"Original predictions: {original_predictions}")
print(f"Loaded model predictions: {loaded_predictions}")
print(f"Predictions match: {np.array_equal(original_predictions, loaded_predictions)}")

# Feature transformation verification
original_transform = ensemble_pipeline[:-1].transform(X_test[:2])
loaded_transform = loaded_model[:-1].transform(X_test[:2])
print(f"Transformations match: {np.allclose(original_transform, loaded_transform)}")

print("Example 5 completed successfully")